# ERA5-Land environment download - Colab version
Outputs are written as one CSV per shapefile ID, variable, statistic, and date range.

In [ ]:
# Colab setup
!pip -q install geemap==0.32.0 pycrs pyshp

In [ ]:
from __future__ import annotations

import glob
import logging
import os
import shutil
import time
from datetime import datetime, timedelta, date
from pathlib import Path

import ee
import geemap
import pandas as pd
import zipfile

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

## 1. Mount Google Drive and authenticate Earth Engine

If your shapefiles and outputs are not stored on Google Drive, set `MOUNT_DRIVE = False` in the next cell and adjust the paths in the parameter cell.


In [ ]:
MOUNT_DRIVE = True

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

In [ ]:
# If Earth Engine asks for a project, set PROJECT_ID to your Google Cloud project ID.
# Example: PROJECT_ID = 'aiv-gee'
PROJECT_ID = "aiv-gee"

try:
    if PROJECT_ID:
        ee.Initialize(project=PROJECT_ID)
    else:
        ee.Initialize()
except Exception:
    ee.Authenticate()
    if PROJECT_ID:
        ee.Initialize(project=PROJECT_ID)
    else:
        ee.Initialize()

print('Earth Engine initialized')

## 2. Parameters

Edit this cell before running the workflow.


In [ ]:
zip_path = Path('/content/drive/MyDrive/gee-download/EU_100km_fishnet_simple_by_distance.zip')
extract_dir = Path('/content/drive/MyDrive/gee-download')

extract_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print('Decompression complete.：', extract_dir)

In [ ]:
ENV_VARS = [
    'dewpoint_temperature_2m',
    'temperature_2m',
    'temperature_2m_min',
    'temperature_2m_max',
    'skin_temperature',
    'soil_temperature_level_1',
    'soil_temperature_level_2',
    'soil_temperature_level_3',
    'soil_temperature_level_4',
    'lake_bottom_temperature',
    'lake_ice_depth',
    'lake_ice_temperature',
    'lake_mix_layer_depth',
    'lake_mix_layer_temperature',
    'lake_shape_factor',
    'lake_total_layer_temperature',
    'snow_albedo',
    'snow_cover',
    'snow_density',
    'snow_depth',
    'snow_depth_water_equivalent',
    'snowfall_sum',
    'snowmelt_sum',
    'temperature_of_snow_layer',
    'skin_reservoir_content',
    'volumetric_soil_water_layer_1',
    'volumetric_soil_water_layer_2',
    'volumetric_soil_water_layer_3',
    'volumetric_soil_water_layer_4',
    'forecast_albedo',
    'surface_latent_heat_flux_sum',
    'surface_net_solar_radiation_sum',
    'surface_net_thermal_radiation_sum',
    'surface_sensible_heat_flux_sum',
    'surface_solar_radiation_downwards_sum',
    'surface_thermal_radiation_downwards_sum',
    'evaporation_from_bare_soil_sum',
    'evaporation_from_open_water_surfaces_excluding_oceans_sum',
    'evaporation_from_the_top_of_canopy_sum',
    'evaporation_from_vegetation_transpiration_sum',
    'potential_evaporation_sum',
    'potential_evaporation_min',
    'potential_evaporation_max',
    'runoff_sum',
    'snow_evaporation_sum',
    'sub_surface_runoff_sum',
    'surface_runoff_sum',
    'total_evaporation_sum',
    'total_evaporation_min',
    'total_evaporation_max',
    'u_component_of_wind_10m',
    'v_component_of_wind_10m',
    'surface_pressure',
    'total_precipitation_sum',
    'total_precipitation_min',
    'total_precipitation_max',
    'leaf_area_index_high_vegetation',
    'leaf_area_index_low_vegetation',
    'Relative_Humidity',
]

STATISTICS = ['MEAN', 'MAXIMUM', 'MINIMUM', 'MEDIAN', 'STD', 'VARIANCE', 'SUM']

In [ ]:
# Input and output locations
SHAPEFILE_ROOT = Path('/content/drive/MyDrive/gee-download')
OUTPUT_ROOT = Path('/content/drive/MyDrive/gee-download/result')
TEMP_DIR = Path('/content/data_all')

# Time range, inclusive
START_DATE = '2020-01-01'
END_DATE = '2020-12-31'

# Choose shapefiles to run.
# None = all folders under SHAPEFILE_ROOT.
# Or use a list, for example: FILE_IDS = ['J001', 'J002']
FILE_IDS = ['EU_100km_fishnet_simple_by_distance']

# Optional index range after sorting discovered shapefile IDs.
# This reproduces the original notebook's files_J[9:10] behavior if set to (9, 10).
FILE_INDEX_RANGE = None

ID_COLUMN = 'Id'
SCALE_METERS = 1000
SKIP_EXISTING_OUTPUT = False
SLEEP_SECONDS_BETWEEN_EXPORTS = 0

ENV_VARS = [
    'dewpoint_temperature_2m',
    'temperature_2m',
]

STATISTICS = ['MEAN']

## 3. Helper functions


In [ ]:
def parse_date(value) -> date:
    if isinstance(value, date):
        return value
    for fmt in ('%Y-%m-%d', '%Y/%m/%d', '%Y%m%d', '%Y_%m_%d'):
        try:
            return datetime.strptime(str(value), fmt).date()
        except ValueError:
            pass
    raise ValueError(f'Unparsable date format: {value}')


def last_day_of_month(any_day: date) -> date:
    next_month = any_day.replace(day=28) + timedelta(days=4)
    return next_month - timedelta(days=next_month.day)


def month_ranges(begin, end):
    begin = parse_date(begin)
    end = parse_date(end)
    if begin > end:
        raise ValueError('START_DATE must be earlier than or equal to END_DATE')

    result = []
    current = begin
    while current <= end:
        month_end = min(last_day_of_month(current), end)
        result.append((current, month_end))
        current = month_end + timedelta(days=1)
    return result


def format_date(value, output_format='%Y/%m/%d'):
    return parse_date(value).strftime(output_format)


def get_relative_humidity(image):
    rh = image.expression(
        '100 * (exp((17.625 * Td)/(243.04 + Td))/exp((17.625*T)/(243.04 + T)))',
        {
            'Td': image.select('dewpoint_temperature_2m').add(-273.15),
            'T': image.select('temperature_2m').add(-273.15),
        },
    ).rename('Relative_Humidity')
    return image.addBands(rh)


def discover_shapefiles(root: Path, file_ids=None, index_range=None):
    root = Path(root)
    if not root.exists():
        raise FileNotFoundError(f'SHAPEFILE_ROOT does not exist: {root}')

    if file_ids is None:
        ids = sorted([p.name for p in root.iterdir() if p.is_dir()])
    else:
        ids = list(file_ids)

    if index_range is not None:
        start, stop = index_range
        ids = ids[start:stop]

    records = []
    missing = []
    for file_id in ids:
        shp_path = root / file_id / f'{file_id}.shp'
        if shp_path.exists():
            records.append((file_id, shp_path))
        else:
            missing.append(str(shp_path))

    if missing:
        logging.warning('Missing shapefiles:\n%s', '\n'.join(missing))
    if not records:
        raise FileNotFoundError('No shapefiles found. Check SHAPEFILE_ROOT, FILE_IDS, and FILE_INDEX_RANGE.')
    return records


def safe_name(value):
    return str(value).replace('/', '-').replace(' ', '_').replace(':', '-')

In [ ]:
def era5_collection_for_period(states, variable, start_day, end_day):
    collection = (
        ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR')
        .filter(ee.Filter.date(
            datetime.combine(start_day, datetime.min.time()),
            datetime.combine(end_day + timedelta(days=1), datetime.min.time()),
        ))
        .map(get_relative_humidity)
        .map(lambda image: image.select([variable]))
        .map(lambda image: image.clip(states))
        .map(lambda image: image.reproject(crs='EPSG:4326', scale=SCALE_METERS))
    )
    return collection


def tidy_zonal_statistics(raw_csv, variable, fallback_day, id_column=ID_COLUMN):
    raw = pd.read_csv(raw_csv)
    if raw.empty:
        return pd.DataFrame(columns=['Date', 'Doy', variable, id_column])
    if id_column not in raw.columns:
        raise KeyError(f'Expected ID column {id_column!r} in {raw_csv}. Available columns: {list(raw.columns)}')

    rows = []
    stat_like_columns = {'mean', 'max', 'maximum', 'min', 'minimum', 'median', 'std', 'variance', 'sum'}

    for column in raw.columns:
        if column == id_column or column.lower() in {'system:index', '.geo'}:
            continue

        column_text = str(column)
        prefix = column_text[:8]
        if prefix.isdigit():
            day = parse_date(prefix)
        elif column_text.lower() in stat_like_columns or column_text == variable:
            day = parse_date(fallback_day)
        else:
            continue

        temp = raw[[id_column, column]].copy()
        temp.insert(0, 'Date', format_date(day))
        temp.insert(1, 'Doy', day.strftime('%j'))
        temp = temp.rename(columns={column: variable})
        rows.append(temp[['Date', 'Doy', variable, id_column]])

    if not rows:
        raise ValueError(f'Could not find data columns for {variable} in {raw_csv}. Columns: {list(raw.columns)}')

    return pd.concat(rows, ignore_index=True)


def run_one_output(file_id, shp_path, variable, statistic, month_list):
    output_dir = OUTPUT_ROOT / file_id
    output_dir.mkdir(parents=True, exist_ok=True)

    output_csv = output_dir / f'{variable}_{statistic}_{START_DATE}_to_{END_DATE}.csv'
    if SKIP_EXISTING_OUTPUT and output_csv.exists():
        logging.info('Skip existing output: %s', output_csv)
        return output_csv

    temp_work_dir = TEMP_DIR / safe_name(file_id) / safe_name(variable) / safe_name(statistic)
    if temp_work_dir.exists():
        shutil.rmtree(temp_work_dir)
    temp_work_dir.mkdir(parents=True, exist_ok=True)

    states = geemap.shp_to_ee(str(shp_path))
    monthly_frames = []

    for start_day, end_day in month_list:
        collection = era5_collection_for_period(states, variable, start_day, end_day)
        image_count = collection.size().getInfo()
        if image_count == 0:
            logging.warning('No ERA5 images found: %s %s to %s', variable, start_day, end_day)
            continue

        stacked = collection.toBands()
        temp_csv = temp_work_dir / f'Era5_{statistic}_{start_day}_to_{end_day}.csv'

        geemap.zonal_statistics(
            stacked,
            states,
            str(temp_csv),
            statistics_type=statistic,
            scale=SCALE_METERS,
        )

        tidy = tidy_zonal_statistics(temp_csv, variable, start_day, ID_COLUMN)
        monthly_frames.append(tidy)

        if SLEEP_SECONDS_BETWEEN_EXPORTS:
            time.sleep(SLEEP_SECONDS_BETWEEN_EXPORTS)

    if not monthly_frames:
        logging.warning('No data written for %s / %s / %s', file_id, variable, statistic)
        return None

    merged = pd.concat(monthly_frames, ignore_index=True)
    merged = merged.rename(columns={variable: f'{variable}_{statistic}'})
    merged.to_csv(output_csv, index=False)
    return output_csv

## 4. Preview workload


In [ ]:
shapefiles = discover_shapefiles(SHAPEFILE_ROOT, FILE_IDS, FILE_INDEX_RANGE)
time_windows = month_ranges(START_DATE, END_DATE)

print(f'Shapefiles: {len(shapefiles)}')
print(f'Month windows: {len(time_windows)}')
print(f'Variables: {len(ENV_VARS)}')
print(f'Statistics: {len(STATISTICS)}')
print(f'Total output CSV targets: {len(shapefiles) * len(ENV_VARS) * len(STATISTICS)}')
print('First shapefiles:', [item[0] for item in shapefiles[:5]])
print('First month windows:', time_windows[:3])

## 5. Run download and zonal statistics



In [ ]:
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

written_files = []
failed_jobs = []

for file_id, shp_path in shapefiles:
    logging.info('Running shapefile %s', file_id)
    for variable in ENV_VARS:
        for statistic in STATISTICS:
            try:
                output_csv = run_one_output(file_id, shp_path, variable, statistic, time_windows)
                if output_csv is not None:
                    written_files.append(output_csv)
                    logging.info('Wrote %s', output_csv)
            except Exception as exc:
                logging.exception('Failed: file_id=%s variable=%s statistic=%s', file_id, variable, statistic)
                failed_jobs.append({
                    'file_id': file_id,
                    'variable': variable,
                    'statistic': statistic,
                    'error': repr(exc),
                })

print(f'Finished. Wrote {len(written_files)} files. Failed jobs: {len(failed_jobs)}')